In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from mlflow.sklearn import load_model

"""
1. 데이터 로딩
2. 고객 이탈 예측을 위한 데이터
3. 'Exited' 컬럼이 예측 대상(target)
"""
data = pd.read_csv('../data/churn.csv')



In [2]:
# 불필요한 컬럼 제거 후 feature(X), target(y) 분리
X = data.drop(['Exited', 'RowNumber', 'CustomerId', 'Surname'], axis=1)
y = data['Exited']

In [3]:
# 범주형 수치형 feature 정의
categorical_features = ['Geography', 'Gender']
numeric_features = ['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary']

In [4]:
# 전처리 파이프라인 설정
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features), # 수치형 데이터 표준화
        ('cat', OneHotEncoder(), categorical_features) # 범주형 데이터 원핫인코딩
    ])

In [5]:
# 데이터 전처리 수행
X_processed = preprocessor.fit_transform(X)

In [6]:
# 데이터셋 학습용과 테스트용으로 분할
X_train, X_test, y_train, y_test = train_test_split(X_processed, y, test_size=0.2, random_state=42)

In [7]:
# 로컬에서 모델 불러오기
loaded_model = load_model("./my_model")
preds = loaded_model.predict(X_test)

# (옵션) Tracking 서버 에서 모델 불러오고 싶은 경우
# model_uri = "runs:/ba0aa44aec5b41f98d24e052de1bd1e1/xgboost_model"
# model = mlflow.xgboost.load_model(model_uri)
# preds = model.predict(X_test)

In [8]:
# 예측 결과 출력
print("Loaded model predictions:", preds[:20])  # 처음 10개 출력

# 성능 평가 지표 계산
acc = accuracy_score(y_test, preds)
precision = precision_score(y_test, preds)
recall = recall_score(y_test, preds)
f1 = f1_score(y_test, preds)
roc_auc = roc_auc_score(y_test, preds)

# 성능 지표 재확인 (선택 사항)
print("Accuracy (loaded model):", acc)
print("precision (loaded model):", precision)
print("recall (loaded model):", recall)
print("f1 (loaded model):", f1)
print("roc_auc (loaded model):", roc_auc)


Loaded model predictions: [0 0 0 0 0 0 0 0 0 0 1 1 1 0 0 0 0 0 0 0]
Accuracy (loaded model): 0.858
precision (loaded model): 0.693950177935943
recall (loaded model): 0.4961832061068702
f1 (loaded model): 0.5786350148367952
roc_auc (loaded model): 0.7213336690148539
